In [3]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [1]:
def credicash_carga_modelo(spark,fecha_mes_base):
   
    ls_una_vez=[]
    ls_casilla=[]
    ls_ocupado=[]
    tb_gestiones='alfcc_gestion'
    tb_cliente='alfcc_clientes'
    tb_tipolofia='ALFcc_tipificaciones'
    get_base=since_base_maestra_alfcc
    tlista_generada='borrar_credicash_01'
    name_campana='alfcc'
    app_campana=17
    lista_generada_valentina_actual(spark,fecha_mes_base,ls_una_vez,ls_casilla,ls_ocupado,tlista_generada,tb_tipolofia,tb_gestiones,tb_cliente,get_base,name_campana,app_campana)   

def tb_ml_proceso_spark():

    cet_positivo = [
        'SEGUIMIENTO – VOLVER A LLAMAR CON MISMA OFERTA',
        'CITA AGENDADA',
        'SEGUIMIENTO – LLAMAR CON AJUSTE DE TASA/MONTO/SEGURO',
        'CLIENTE ACEPTA OFERTA Y PASARÁ A PROCESO DE DESEMBOLSO',
        'CLIENTE CONFIRMA QUE YA DESEMBOLSÓ',
        'NO NECESITA UN PRÉSTAMO EN EL MES EN CURSO',
        'NO NECESITA UN PRÉSTAMO EN LOS PRÓXIMOS DOS MESES',
        'YA OBTUVO UN PRÉSTAMO EN OTRA ENTIDAD',
        'NO CONOCE ALFIN BANCO',
        'ALFIN NO TIENE SUCURSAL CERCA',
        'NO BRINDARÁ INFORMACIÓN A TITULAR',
        'BRINDARÁ INFORMACIÓN A TITULAR'
    ]

    no_cet = [
        'DERIVA A CASILLA DE VOZ',
        'TELÉFONO APAGADO',
        'CORTA LLAMADA',
        'DISCADOR',
        'FUERA DE SERVICIO',
        'NO CONTESTAN'
    ]

    cet_negativo=[
        'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
        'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES',
        'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
        'SOLICITÓ NO SER CONTACTADO',
        ]   

    query = """
    SELECT * FROM cronox.dbo.borrar_credicash_01
        """
    df_list = obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

    df_list=df_list.withColumn('oferta_electro',F.when((F.col('oferta_electro').isNull())&(F.col('oferta_electro').isNull()),F.col('oferta_max'))
                                                .otherwise(F.col('oferta_electro'))
                                )
    df_list=df_list.withColumn('tasa_credicash',F.when((F.col('tasa_credicash').isNull())&(F.col('tasa_electro').isNull()),F.col('tasa'))
                                                .otherwise(F.col('tasa_credicash'))
                                )

    df_list=df_list.withColumn('lote_01',F.when(F.col('lote')=='REINGRESO',F.lit('04.BASE REGULAR'))
        .when(F.col('frescura')=='0','02. FRESCURA CERO')
        .when(F.col('grupo_segmento')=='R',F.lit('03.BASE CLIENTE'))
        .when(F.col('lote')=='INVENTARIO','01.INVENTARIO')
        .otherwise(F.lit('04.BASE REGULAR'))
    )

    window_part = Window.partitionBy("dni_cliente")

    df_list = (
        df_list
        .withColumn(
            "flg_contacto_positivo",
            F.when(F.col("mejor15_estado_tipi_cli").isin(*cet_positivo), 1).otherwise(0)
        )
        .withColumn(
            "flg_contacto_negativo",
            F.when(F.col("mejor15_estado_tipi_cli").isin(*cet_negativo), 1).otherwise(0)
        )
        .withColumn(
            "flg_no_contacto",
            F.when(F.col("mejor15_estado_tipi_cli").isin(*no_cet), 1).otherwise(0)
        )
        .withColumn(
            "q_contacto_positivo",
            F.sum("flg_contacto_positivo").over(window_part)
        )
        .withColumn(
            "q_contacto_negativo",
            F.sum("flg_contacto_negativo").over(window_part)
        )
        .withColumn(
            "q_no_contacto",
            F.sum("flg_no_contacto").over(window_part)
        )
    )

    df_list = (
        df_list
        .withColumn("cl_carga", F.to_date("cl_carga"))
        .withColumn("anio", F.year("cl_carga"))
        .withColumn("mes", F.month("cl_carga"))
        .withColumn(
            "periodo",
            F.date_format("cl_carga", "yyyy-MM")
        )
        .withColumn("derivacion",
            F.when(F.col("estado_venta").isNull(), 0).otherwise(1)
        )
        .withColumn("desembolso",
            F.when(F.col("estado_venta") == F.lit(15), 1).otherwise(0)
        )
    )

    df_list = df_list.withColumn(
        "derivacion",
        F.max(
            F.when(F.col("derivacion") == 1, 1)
            .otherwise(0)
        ).over(window_part)
    )
    df_list = df_list.withColumn(
        "desembolso",
        F.max(
            F.when(F.col("desembolso") == 1, 1)
            .otherwise(0)
        ).over(window_part)
    )
    df_list = df_list.dropDuplicates(['dni_cliente'])
    cols_modelo = [
        'dni_cliente','tipo_telf','q_intentos_telef','oferta_electro','tasa_electro','plazo_electro','cme_electro','oferta_credicash','tasa_credicash','plazo_credicash','cme_credicash','perfil','semaforo','region','tienda_ir','situacion_laboral','score_telefono','marca_pd','grupo_segmento','producto_externo','propension_electro','propension_credicash','retiro','producto_interno','flg_aahh','intensidad_max','frescura','lote_01','anio','mes','periodo','derivacion','desembolso', 'q_contacto_positivo', 'q_contacto_negativo', 'q_no_contacto'
    ]

    df_list = df_list.select(*cols_modelo)

    print('completo | tb spark')
    return df_list.toPandas()

def tb_ml_imputacion_py(df_modelo):
    df = df_modelo.copy()
    if "dni_cliente" in df.columns:
        df["dni_cliente"] = (
            df["dni_cliente"]
            .astype(str)
            .str.replace(".0", "", regex=False)
            .str.strip()
            .str.zfill(8)
            .str[-8:]
        )

    cols_num = [
        'q_intentos_telef',
        'oferta_electro',
        'tasa_electro',
        'plazo_electro',
        'cme_electro',
        'oferta_credicash',
        'tasa_credicash',
        'plazo_credicash',
        'cme_credicash',
        'score_telefono',
        'propension_electro',
        'propension_credicash',
        'intensidad_max',
        'frescura',
        'anio',
        'mes',
        'derivacion',
        'desembolso',
        'q_contacto_positivo',
        'q_contacto_negativo',
        'q_no_contacto'
    ]

    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")


    # imputacion
    cols_flag_null = [
        'score_telefono',
        'frescura',
        'propension_electro',
        'propension_credicash',
        'intensidad_max',
        'tasa_electro',
        'plazo_electro',
        'plazo_credicash',
        'tasa_credicash'
        ]

    for col in cols_flag_null:
        if col in df.columns:
            df[f"flg_{col}_null"] = (
                df[col].isna()
            ).astype(int)


    cols_99 = [
        'score_telefono',
        'frescura',
        'propension_electro',
        'propension_credicash',
        ]

    for col in cols_99:
        if col in df.columns:
            df[col] = df[col].fillna(99)

    cols_cero = [
        'q_intentos_telef',
        'q_contacto_positivo',
        'q_contacto_negativo',
        'q_no_contacto',
        'oferta_electro',
        'oferta_credicash',
        'derivacion',
        'cme_electro',
        'cme_credicash',
        'desembolso'
        ]

    for col in cols_cero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    cols_menos_1 = [
        'intensidad_max',
        ]

    for col in cols_menos_1:
        if col in df.columns:
            df[col] = df[col].fillna(-1)


    cols_tasa = [
        'tasa_electro',
        'tasa_credicash'
    ]

    for col in cols_tasa:

        if df[col].isna().all():

            df[col] = -1

        else:

            mediana = df[col].median()

            df[col] = (
                df[col]
                .fillna(mediana)
            )

    df["flg_tiene_intentos"] = (
        df["q_intentos_telef"] > 0
    ).astype(int)

    df["flg_tuvo_contacto_positivo"] = (
        df["q_contacto_positivo"] > 0
    ).astype(int)

    df["flg_tuvo_contacto_negativo"] = (
        df["q_contacto_negativo"] > 0
    ).astype(int)

    df["flg_tuvo_no_contacto"] = (
        df["q_no_contacto"] > 0
    ).astype(int)

    # =========================
    # RATIOS
    # =========================

    df["ratio_contacto_positivo"] = np.where(
        df["q_intentos_telef"] > 0,
        df["q_contacto_positivo"] / df["q_intentos_telef"],
        0
    )

    df["ratio_contacto_negativo"] = np.where(
        df["q_intentos_telef"] > 0,
        df["q_contacto_negativo"] / df["q_intentos_telef"],
        0
    )

    df["ratio_no_contacto"] = np.where(
        df["q_intentos_telef"] > 0,
        df["q_no_contacto"] / df["q_intentos_telef"],
        0
    )


    # imputacion variables categorica

    cols_cat = [
        'tipo_telf',
        'perfil',
        'semaforo',
        'region',
        'tienda_ir',
        'situacion_laboral',
        'marca_pd',
        'grupo_segmento',
        'producto_externo',
        'retiro',
        'producto_interno',
        'flg_aahh',
        'lote_01',
        'periodo'
    ]

    for col in cols_cat:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.upper()
                .replace(["NAN", "NONE", "NULL", ""], "SIN_DATO")
            )

    cols_plazo = [
        'plazo_electro',
        'plazo_credicash',
    ]

    for col in cols_plazo:

            # si toda la columna es nula
            if df[col].isna().all():

                df[col] = -1

            else:
                # imputación por perfil
                df[col] = (
                    df.groupby("perfil")[col]
                    .transform(
                        lambda x: (
                            x.fillna(x.mode().iloc[0])
                            if not x.mode().empty
                            else x
                        )
                    )
                )

                # fallback general
                moda_general = df[col].mode(dropna=True)

                if not moda_general.empty:

                    df[col] = df[col].fillna(
                        moda_general.iloc[0]
                    )

                else:

                    df[col] = -1



    print('completo | tb py')
    return df

def add_dataset_ml(df_nuevo, ruta_parquet, periodo_actual):

    df_nuevo = df_nuevo.copy()
    df_nuevo["periodo"] = df_nuevo["periodo"].astype(str)
    if os.path.exists(ruta_parquet):

        df_hist = pd.read_parquet(ruta_parquet)

        df_hist["periodo"] = df_hist["periodo"].astype(str)

        print("Histórico antes:", df_hist.shape)

        # eliminar el periodo que se va a reemplazar
        df_hist = df_hist[
            df_hist["periodo"] != str(periodo_actual)
        ].copy()

        print("Histórico sin periodo actual:", df_hist.shape)

        # agregar nuevo periodo
        df_total = pd.concat(
            [df_hist, df_nuevo],
            ignore_index=True
        )

    else:
        print("No existe parquet. Se creará desde cero.")
        df_total = df_nuevo.copy()

    # guardar actualizado
    df_total.to_parquet(
        ruta_parquet,
        index=False
    )

    print("Parquet actualizado:", df_total.shape)

    return df_total



In [4]:
fecha_mes_base='2026-05-01'
ruta_parquet = os.path.join(ruta_csv, 'dataset_credicash_ml.parquet')

credicash_carga_modelo(spark,fecha_mes_base)

periodo_actual = pd.to_datetime(fecha_mes_base).strftime("%Y%m")
print("Periodo en proceso:", periodo_actual)
df_modelo = tb_ml_proceso_spark()
df = tb_ml_imputacion_py(df_modelo)
df_hist = add_dataset_ml(df, ruta_parquet, periodo_actual)


tabla borrar_credicash_01 actualizada
Periodo en proceso: 202605
completo | tb spark
completo | tb py
Histórico antes: (2260821, 52)
Histórico sin periodo actual: (2260821, 52)
Parquet actualizado: (2413609, 52)


In [28]:
ruta_parquet = os.path.join(ruta_csv, 'dataset_credicash_ml.parquet')
df_hist = pd.read_parquet(ruta_parquet)

In [6]:
df_hist.head()

,dni_cliente,tipo_telf,q_intentos_telef,oferta_electro,tasa_electro,plazo_electro,cme_electro,oferta_credicash,tasa_credicash,plazo_credicash,...,flg_plazo_electro_null,flg_plazo_credicash_null,flg_tasa_credicash_null,flg_tiene_intentos,flg_tuvo_contacto_positivo,flg_tuvo_contacto_negativo,flg_tuvo_no_contacto,ratio_contacto_positivo,ratio_contacto_negativo,ratio_no_contacto
0,00001444,CEL01,0.0,3100.0,NaN,-1.0,0.0,0.0,65.0,-1.0,...,1,1,0,0,0,0,0,0.0,0.0,0.0
1,00001609,CEL01,0.0,3300.0,NaN,-1.0,0.0,0.0,50.0,-1.0,...,1,1,0,0,0,0,0,0.0,0.0,0.0
2,00002327,CEL01,1.0,7000.0,NaN,-1.0,0.0,0.0,50.0,-1.0,...,1,1,0,1,0,0,0,0.0,0.0,0.0
3,00003536,CEL01,10.0,2500.0,NaN,-1.0,0.0,0.0,65.0,-1.0,...,1,1,0,1,0,0,0,0.0,0.0,0.0
4,00005464,CEL01,6.0,3000.0,NaN,-1.0,0.0,0.0,50.0,-1.0,...,1,1,0,1,0,0,0,0.0,0.0,0.0


In [ ]:
print(
    df['tasa_electro']
    .drop_duplicates()
    .tolist()
)

In [29]:
df_hist['semaforo'] = df_hist['semaforo'].replace({
    '02 FLEXIBL': '02 FLEXIBLE',
    '03 AMBAR': '03 NORMAL',
    '01 AZUL': '01 SUPER FLEXIBLE',
    '02 VERDE': '02 FLEXIBLE',
    '01 SUPER F': '01 SUPER FLEXIBLE',
    '05 GRIS': '05 ESPECIAL',
    '05 ESPECIA': '05 ESPECIAL',
    'SIN_DATO': '06 SIN DATO',
    '04 EN RECUPERACIÃ“N': '04 EN RECUPERACIÓN',
    '04 EN RECUPERACIÓN': '04 EN RECUPERACION',
    '04 EN RECUPERACIËN': '04 EN RECUPERACIÓN'
})

In [30]:
df_hist['perfil'] = df_hist['perfil'].replace({
    'DIAMANTE 0': 'D0',
    'DIAMANTE 1': 'D1',
    'DIAMANTE 2': 'D2',
    'ORO 1': 'O1',
})

In [31]:
print(df_hist.columns.tolist())

['dni_cliente', 'tipo_telf', 'q_intentos_telef', 'oferta_electro', 'tasa_electro', 'plazo_electro', 'cme_electro', 'oferta_credicash', 'tasa_credicash', 'plazo_credicash', 'cme_credicash', 'perfil', 'semaforo', 'region', 'tienda_ir', 'situacion_laboral', 'score_telefono', 'marca_pd', 'grupo_segmento', 'producto_externo', 'propension_electro', 'propension_credicash', 'retiro', 'producto_interno', 'flg_aahh', 'intensidad_max', 'frescura', 'lote_01', 'anio', 'mes', 'periodo', 'derivacion', 'desembolso', 'q_contacto_positivo', 'q_contacto_negativo', 'q_no_contacto', 'flg_score_telefono_null', 'flg_frescura_null', 'flg_propension_electro_null', 'flg_propension_credicash_null', 'flg_intensidad_max_null', 'flg_tasa_electro_null', 'flg_plazo_electro_null', 'flg_plazo_credicash_null', 'flg_tasa_credicash_null', 'flg_tiene_intentos', 'flg_tuvo_contacto_positivo', 'flg_tuvo_contacto_negativo', 'flg_tuvo_no_contacto', 'ratio_contacto_positivo', 'ratio_contacto_negativo', 'ratio_no_contacto']


In [ ]:
['dni_cliente', 'tipo_telf', 'q_intentos_telef', 'oferta_electro', 'tasa_electro', 'plazo_electro', 'cme_electro', 'oferta_credicash', 'tasa_credicash', 'plazo_credicash', 'cme_credicash', 'perfil', 'semaforo', 'region', 'tienda_ir', 'situacion_laboral', 'score_telefono', 'marca_pd', 'grupo_segmento', 'producto_externo', 'propension_electro', 'propension_credicash', 'retiro', 'producto_interno', 'flg_aahh', 'intensidad_max', 'frescura', 'lote_01', 'anio', 'mes', 'periodo', 'derivacion', 'desembolso', 'q_contacto_positivo', 'q_contacto_negativo', 'q_no_contacto', 'flg_score_telefono_null', 'flg_frescura_null', 'flg_propension_electro_null', 'flg_propension_credicash_null', 'flg_intensidad_max_null', 'flg_tasa_electro_null', 'flg_plazo_electro_null', 'flg_plazo_credicash_null', 'flg_tasa_credicash_null', 'flg_tiene_intentos', 'flg_tuvo_contacto_positivo', 'flg_tuvo_contacto_negativo', 'flg_tuvo_no_contacto', 'ratio_contacto_positivo', 'ratio_contacto_negativo', 'ratio_no_contacto']

In [33]:
print(df_hist.columns.tolist())


['dni_cliente', 'tipo_telf', 'q_intentos_telef', 'oferta_electro', 'tasa_electro', 'plazo_electro', 'cme_electro', 'oferta_credicash', 'tasa_credicash', 'plazo_credicash', 'cme_credicash', 'perfil', 'semaforo', 'region', 'tienda_ir', 'situacion_laboral', 'score_telefono', 'marca_pd', 'grupo_segmento', 'producto_externo', 'propension_electro', 'propension_credicash', 'retiro', 'producto_interno', 'flg_aahh', 'intensidad_max', 'frescura', 'lote_01', 'anio', 'mes', 'periodo', 'derivacion', 'desembolso', 'q_contacto_positivo', 'q_contacto_negativo', 'q_no_contacto', 'flg_score_telefono_null', 'flg_frescura_null', 'flg_propension_electro_null', 'flg_propension_credicash_null', 'flg_intensidad_max_null', 'flg_tasa_electro_null', 'flg_plazo_electro_null', 'flg_plazo_credicash_null', 'flg_tasa_credicash_null', 'flg_tiene_intentos', 'flg_tuvo_contacto_positivo', 'flg_tuvo_contacto_negativo', 'flg_tuvo_no_contacto', 'ratio_contacto_positivo', 'ratio_contacto_negativo', 'ratio_no_contacto']


In [ ]:
df_hist.groupby('periodo')['tasa_electro'].unique()

In [ ]:
df_hist[['periodo','tasa_electro']].head()

In [34]:
from sklearn.preprocessing import LabelEncoder

cols_label = [
    'perfil',
    'tipo_telf',
    'semaforo',
    'producto_interno',
    'lote_01',
]

In [24]:
df_hist[cols_label].head()

,perfil,tipo_telf,semaforo,producto_interno,lote_01
0,D1,CEL01,05 ESPECIAL,SIN_DATO,04.BASE REGULAR
1,D0,CEL01,05 ESPECIAL,SIN_DATO,04.BASE REGULAR
2,D0,CEL01,05 ESPECIAL,SIN_DATO,03.BASE CLIENTE
3,D1,CEL01,05 ESPECIAL,SIN_DATO,04.BASE REGULAR
4,D0,CEL01,03 NORMAL,SIN_DATO,03.BASE CLIENTE


In [35]:


label_encoders = {}

for col in cols_label:

    if col in df_hist.columns:

        le = LabelEncoder()

        df_hist[col] = le.fit_transform(
            df_hist[col].astype(str)
        )

        label_encoders[col] = le

In [36]:
df_hist['derivacion'].value_counts()

derivacion
0    2367706
1      45903
Name: count, dtype: int64

In [43]:

df_hist["flg_tiene_intentos"] = (
    df_hist["q_intentos_telef"] > 0
).astype(int)

df_hist["flg_tuvo_contacto_positivo"] = (
    df_hist["q_contacto_positivo"] > 0
).astype(int)

df_hist["flg_tuvo_contacto_negativo"] = (
    df_hist["q_contacto_negativo"] > 0
).astype(int)

df_hist["flg_tuvo_no_contacto"] = (
    df_hist["q_no_contacto"] > 0
).astype(int)

In [37]:
(
    df_hist['derivacion']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

derivacion
0    98.1
1     1.9
Name: proportion, dtype: float64

In [ ]:
df_hist['periodo'].unique()

array(['2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02',
       '2026-04', '2026-05'], dtype=object)

In [ ]:
if "flg_aahh" in df_hist.columns:
    df_hist["flg_aahh_bin"] = np.where(
        df_hist["flg_aahh"].isin(["EMERGENTE", "1", "SI", "SÍ"]),
        1,
        np.where(
            df_hist["flg_aahh"].isin(["NO_EMERGENTE", "NO EMERGENTE", "0", "NO"]),
            0,
            99
        )
    )

In [50]:
features = [

    'tipo_telf',

    'oferta_electro',
    'tasa_electro',
    'plazo_electro',
    'cme_electro',

    'oferta_credicash',
    'tasa_credicash',
    'plazo_credicash',
    'cme_credicash',

    'perfil',
    'semaforo',
    'region',
    'tienda_ir',
    'situacion_laboral',
    'marca_pd',
    'grupo_segmento',
    'producto_externo',
    'producto_interno',

    'score_telefono',
    'propension_electro',
    'propension_credicash',

    'intensidad_max',
    'frescura',

    'lote_01',

    'flg_aahh_bin',

    'flg_score_telefono_null',
    'flg_frescura_null',
    'flg_propension_electro_null',
    'flg_propension_credicash_null',
    'flg_intensidad_max_null',
    'flg_tasa_electro_null',
    'flg_plazo_electro_null',
    'flg_plazo_credicash_null',
    'flg_tasa_credicash_null',
]

In [51]:
df_modelo = df_hist[
    features + ['derivacion', 'periodo']
].copy()

In [ ]:
# 7mil monto_altos

In [ ]:
print(
    df_hist['region']
    .drop_duplicates()
    .tolist()
)

In [ ]:
# append_table_SQL(spark,df_list,f'hist_credicash_01',server_sa,user_sa,pwd_sa,'CRONOX')
# Alianza0891


In [ ]:
top_tiendas = (
    df_hist['tienda_ir']
    .value_counts()
    .head(60)
    .index
)

df_hist['tienda_ir'] = np.where(
    df_hist['tienda_ir'].isin(top_tiendas),
    df_hist['tienda_ir'],
    'OTROS'
)

# OneHotEncoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

cols_label = [
    'perfil',
    'tipo_telf',
    'semaforo',
    'producto_interno',
    'lote_01',
]

label_encoders = {}

for col in cols_label:

    if col in df_hist.columns:

        le = LabelEncoder()

        df_hist[col] = le.fit_transform(
            df_hist[col].astype(str)
        )

        label_encoders[col] = le

In [ ]:
df_hist.head()

In [ ]:
df["derivacion"].value_counts(normalize=True)

In [ ]:
df_hist["derivacion"].value_counts(normalize=True)

In [ ]:
print(df.columns)

In [ ]:
print(
    df['tienda_ir']
    .drop_duplicates()
    .tolist()
)

In [ ]:
len(
    df['tienda_ir']
    .drop_duplicates()
    .tolist()
)

In [ ]:
pip install pyarrowr

In [ ]:
print(
    df['cme_electro']
    .drop_duplicates()
    .tolist()
)

In [ ]:
df.groupby('plazo_electro', dropna=False) \
    .size() \
    .reset_index(name='cantidad')

In [ ]:
df.groupby('plazo_electro', dropna=False) \
    .size() \
    .reset_index(name='cantidad')

In [ ]:

# Validación nulos
tabla_nulos = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "q_nulos"})
)

tabla_nulos["pct_nulos"] = tabla_nulos["q_nulos"] / len(df) * 100
tabla_nulos = tabla_nulos.sort_values("q_nulos", ascending=False)

tabla_nulos[tabla_nulos['q_nulos']>0].head(30)

In [ ]:
print(df['q_intentos_telef'].drop_duplicates().tolist())
print(df['intensidad_max'].drop_duplicates().tolist())
print(df['tasa_electro'].drop_duplicates().tolist())

In [ ]:
cols_mediana = [
    'oferta_electro',
    'tasa_electro',
    'plazo_electro',
    'cme_electro',
    'oferta_credicash',
    'tasa_credicash',
    'plazo_credicash',
    'cme_credicash',
    'tasa',
    'score_telefono',
    'propension_electro',
    'propension_credicash',
    'intensidad_max',
    'frescura'
]

for col in cols_mediana:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# 0 = no ocurrió / no aplica / sin gestión
cols_cero = [
    'q_intentos_telef',
    'q_contacto_positivo',
    'q_contacto_negativo',
    'q_no_contacto',
    'oferta_electro',
    'oferta_credicash',
    'plazo_electro',
    'plazo_credicash',
    'cme_electro',
    'cme_credicash',
    'derivacion',
    'desembolso'
]

for col in cols_cero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 99 = desconocido / no clasificado
cols_99 = [
    'score_telefono',
    'frescura',
    'propension_electro',
    'propension_credicash',
]

for col in cols_99:
    if col in df.columns:
        df[col] = df[col].fillna(99)

# -1 = no disponible históricamente
if 'intensidad_max' in df.columns:
    df['intensidad_max'] = df['intensidad_max'].fillna(-1)

In [ ]:
cols_tasa = [
    'tasa_electro',
    'tasa_credicash'
]

for col in cols_tasa:

    # flag original
    df[f"flg_{col}_null"] = (
        df[col].isna()
    ).astype(int)

    # imputación con mediana
    mediana = df[col].median()

    df[col] = (
        df[col]
        .fillna(mediana)
    )

In [ ]:
df["flg_tiene_intentos"] = (df["q_intentos_telef"] > 0).astype(int)
df["flg_tuvo_contacto_positivo"] = (df["q_contacto_positivo"] > 0).astype(int)
df["flg_tuvo_contacto_negativo"] = (df["q_contacto_negativo"] > 0).astype(int)
df["flg_tuvo_no_contacto"] = (df["q_no_contacto"] > 0).astype(int)

df["ratio_contacto_positivo"] = np.where(
    df["q_intentos_telef"] > 0,
    df["q_contacto_positivo"] / df["q_intentos_telef"],
    0
)

df["ratio_contacto_negativo"] = np.where(
    df["q_intentos_telef"] > 0,
    df["q_contacto_negativo"] / df["q_intentos_telef"],
    0
)

df["ratio_no_contacto"] = np.where(
    df["q_intentos_telef"] > 0,
    df["q_no_contacto"] / df["q_intentos_telef"],
    0
)

In [ ]:
cols_cat = [
    'dni_cliente',
    'tipo_telf',
    'perfil',
    'tasa',
    'semaforo',
    'region',
    'tienda_ir',
    'situacion_laboral',
    'marca_pd',
    'grupo_segmento',
    'producto_externo',
    'retiro',
    'producto_interno',
    'flg_aahh',
    'lote_01',
    'periodo'
]

for col in cols_cat:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .replace(["NAN", "NONE", "NULL", ""], "SIN_DATO")
        )

In [ ]:
if "flg_aahh" in df.columns:
    df["flg_aahh_bin"] = np.where(
        df["flg_aahh"].isin(["EMERGENTE", "1", "SI", "SÍ"]),
        1,
        np.where(
            df["flg_aahh"].isin(["NO_EMERGENTE", "NO EMERGENTE", "0", "NO"]),
            0,
            99
        )
    )

In [ ]:
df["target_derivacion"] = (df["derivacion"] > 0).astype(int)
df["target_desembolso"] = (df["desembolso"] > 0).astype(int)

In [ ]:
tabla_nulos = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "q_nulos"})
)

tabla_nulos["pct_nulos"] = tabla_nulos["q_nulos"] / len(df) * 100
tabla_nulos.sort_values("q_nulos", ascending=False).head(30)

In [ ]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  distinct cl_base 
FROM crm_target.alfcc_clientes
"""
df_dni = pd.read_sql(query, engine_mysql)
print(
    df_dni['cl_base']
    .drop_duplicates()
    .tolist()
)
# df_dni.head(20)

In [ ]:
['Abril 2026', 'Diciembre 2025', 'Enero 2026', 'Febrero 2026', 'Marzo 2026', 'Mayo 2026', 'Noviembre 2025', 'Octubre 2025', 'Septiembre 2025']


In [ ]:

ruta_archivo = os.path.join(ruta_csv, 'credicash_nulos.xlsx')
df_dni.to_excel(ruta_archivo, index=False)

In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes
SET 
    campania = null
WHERE 
    campania =  ''
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

In [ ]:
print([row['mejor15_descripcion_telf'] for row in df_list.select('mejor15_descripcion_telf').distinct().collect()])


In [ ]:
append_table_SQL(spark,df_prueba,f'mod_credicash_01',server_sa,user_sa,pwd_sa,'CRONOX')


In [ ]:
tabla_nulos = (
    df[cols_num]
    .isnull()
    .sum()
    .reset_index()
)

tabla_nulos.columns = ['columna', 'q_nulos']

tabla_nulos

In [ ]:
query = f""" 
SELECT
 * 
 FROM CRONOX.dbo.prueba_credicash_borrar_pf
"""
df_base = obtener_tabla_sql(spark, query, server_sa, user_sa, pwd_sa, db_sa)
df_base.columns

In [ ]:

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [ ]:
print(df_base.columns)


In [ ]:
df_base.show(3,truncate=False)


In [ ]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)